In [2]:
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import os
import time
import matplotlib.pyplot as plt
from sklearn import metrics
from torch.optim.lr_scheduler import LambdaLR
from tqdm import tqdm
from efficient_kan import KAN
import collections.abc as c

In [3]:
class FunctionalGroupsDataset(Dataset):
    '''
    PyTorch compatible dataset of functional groups files

    Parameters
    ----------

    func_group : str
        Name of the functional group to retrieve data from.
    
    convert_to : str, default None
        Converts all data to specified data type.

    '''
    def __init__(self, func_group: str, convert_to: str = None) -> None:
        self.convert_to = convert_to
        self.func_group_number = np.where(func_group_names.values == func_group)[0][0]
        self.main_dir =  os.path.join('D:\\SHAPE\\1VII\\functional-cnn-main\\ALL')
        self.func_group = func_group

        #NIST IDs that passed preprocessing
        preprocessed_data_dir = pd.DataFrame(os.listdir(os.path.join(self.main_dir, 'preprocessed_data')))


        if func_group not in os.listdir(os.path.join(self.main_dir, 'functional_groups')):
            raise ValueError(f'{func_group} is not present in our database.')
        else:
            #All NIST IDs of specific functional group
            func_group_data_dir = pd.DataFrame(os.listdir(os.path.join(self.main_dir, 'functional_groups', func_group)))

        #NIST IDs of specific functional group that passed preprocessing
        to_sample = pd.merge(preprocessed_data_dir, func_group_data_dir, on = [0, 0], how = 'outer', indicator = True).query('_merge=="left_only"')[0]

        #Equinumerous dataset of preprocessed functional group NIST IDs and shuffled from every other functional groups
        if len(to_sample) < len(func_group_data_dir):
            func_group_data_dir = func_group_data_dir.sample(len(to_sample))
            self.data = pd.concat([to_sample.sample(len(to_sample)), func_group_data_dir], axis = 0)
        else:
            self.data = pd.concat([to_sample.sample(len(func_group_data_dir)), func_group_data_dir], axis = 0)

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        file_path = os.path.join(self.main_dir, 'preprocessed_data', self.data.iloc[index][0])
        file = pd.read_csv(file_path)
        spectra_type = file['spectraType'][0]



        #Converts data
        # if not self.convert_to:
        #     spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        # elif self.convert_to.lower() not in ('absorbance', 'absorbancja', 'transmittance', 'transmitancja'):
        #     raise ValueError(f'Cant convert to {self.convert_to}.')

        # elif self.convert_to.lower() != str(spectra_type).lower():
        #     spectra = torch.nan_to_num(torch.tensor(np.abs(1 - file['y'].values), requires_grad=True)).to(torch.float)
        
        # else:
        #     spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        if not self.convert_to:
            spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        elif self.convert_to.lower() not in ('absorbance', 'absorbancja', 'transmittance', 'transmitancja'):
            raise ValueError(f'Cant convert to {self.convert_to}.')
        
        elif str(spectra_type).lower() not in ('converted_to_absorbance', "absorbance"):
            spectra = torch.nan_to_num(torch.tensor(np.abs(1 - file['y'].values), requires_grad=True)).to(torch.float)
        
        else:
            spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        #Reshapes it as required
        spectra = spectra.reshape(1, 1, 3106)
        
        #Prevents unknown problem with NaN from before
        func_group = torch.nan_to_num(torch.tensor(file['funcGroups'].values[self.func_group_number], requires_grad=True)).to(torch.float)

        return spectra, func_group
    
    def get_nist_id(self, index):
        return self.data.iloc[index][0]

In [4]:
#definition of neural network architecture CNN-KAN, the size and number of KAN layers specified in constructor parameter 
class ChemicyBad(torch.nn.Module):
    def __init__(self, kan_layers: list) -> None:
        super(ChemicyBad, self).__init__()

        self.tr_accuracy = None
        self.te_accuracy = None
        self.tr_loss = None
        self.te_loss = None

        kernel_size_1 = 5
        stride_conv_1 = 1

        stride_pool_1 = 3
        filter_size_1 = 3

        self.conv_1 = torch.nn.Conv1d(1, 10, kernel_size=kernel_size_1, stride=stride_conv_1)

        self.pool_1 = torch.nn.MaxPool1d(filter_size_1, stride_pool_1)

        self.conv_2 = torch.nn.Conv1d(10, 10, kernel_size=kernel_size_1, stride=stride_conv_1)

        self.pool_2 = torch.nn.MaxPool1d(filter_size_1, stride_pool_1)

        self.conv_3 = torch.nn.Conv1d(10, 10, kernel_size=kernel_size_1, stride=stride_conv_1)

        self.pool_3 = torch.nn.MaxPool1d(filter_size_1, stride_pool_1)

        self.kan = KAN([1130] + kan_layers).to(device)

        #self.linear_out = torch.nn.Linear(kan_layers[-1], 1)
        
    def forward(self, x):
        bs, c, h, w = x.shape
        x = x.reshape(bs, c, h*w)
        out = self.conv_1(x)
        out = F.tanh(out)
        out = self.pool_1(out)

        out = self.conv_2(out)
        out = F.tanh(out)
        out = self.pool_2(out)
        
        out = self.conv_3(out)
        out = F.tanh(out)
        out = self.pool_3(out)

        out = out.reshape(x.shape[0],1,out.shape[2]*out.shape[1])

        out = self.kan(out)
        #out = self.linear_out(out)
        out = F.sigmoid(out)
        return out.squeeze(1)

    


NameError: name 'torch' is not defined

In [4]:
# Creating a dataset based on the ENFSI database.
# NOTE: File paths are specified in the class definition and may need to be adjusted.

class FG3Dataset(Dataset):
    '''
    PyTorch compatible dataset of functional groups files

    Parameters
    ----------

    func_group : str
        Name of the functional group to retrieve data from.
    
    convert_to : str, default None
        Converts all data to specified data type.

    '''
    def __init__(self, func_group: str, convert_to: str = None) -> None:
        self.convert_to = convert_to
        self.func_group_number = np.where(func_group_names.values == func_group)[0][0]
        self.main_dir =  os.path.join('C:\\Users\\Tomek_\\Desktop\\cc')
        self.func_group = func_group

        #NIST IDs that passed preprocessing
        preprocessed_data_dir = pd.DataFrame(os.listdir(os.path.join(self.main_dir, 'preprocessed_ENFSI')))


        if func_group not in os.listdir(os.path.join(self.main_dir, 'FGYES')):
            raise ValueError(f'{func_group} is not present in our database.')
        else:
            #All NIST IDs of specific functional group
            func_group_data_dir = pd.DataFrame(os.listdir(os.path.join(self.main_dir, 'FGYES', func_group)))

        #NIST IDs of specific functional group that passed preprocessing
        to_sample = pd.merge(preprocessed_data_dir, func_group_data_dir, on = [0, 0], how = 'outer', indicator = True).query('_merge=="left_only"')[0]

        #Equinumerous dataset of preprocessed functional group NIST IDs and shuffled from every other functional groups
        if len(to_sample) < len(func_group_data_dir):
            func_group_data_dir = func_group_data_dir.sample(len(to_sample))
            self.data = pd.concat([to_sample.sample(len(to_sample)), func_group_data_dir], axis = 0)
        else:
            self.data = pd.concat([to_sample.sample(len(func_group_data_dir)), func_group_data_dir], axis = 0)

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        file_path = os.path.join(self.main_dir, 'preprocessed_ENFSI', self.data.iloc[index][0])
        file = pd.read_csv(file_path)
        spectra_type = file['spectraType'][0]



        #Converts data
        # if not self.convert_to:
        #     spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        # elif self.convert_to.lower() not in ('absorbance', 'absorbancja', 'transmittance', 'transmitancja'):
        #     raise ValueError(f'Cant convert to {self.convert_to}.')

        # elif self.convert_to.lower() != str(spectra_type).lower():
        #     spectra = torch.nan_to_num(torch.tensor(np.abs(1 - file['y'].values), requires_grad=True)).to(torch.float)
        
        # else:
        #     spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        if not self.convert_to:
            spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)

        elif self.convert_to.lower().strip() not in ('absorbance', 'absorbancja', 'transmittance', 'transmitancja'):
            raise ValueError(f'Cant convert to {self.convert_to}.')
        
        elif str(spectra_type).lower().strip() not in ('converted_to_absorbance', "absorbance", ):
            spectra = torch.nan_to_num(torch.tensor(np.abs(1 - file['y'].values), requires_grad=True)).to(torch.float)
            #raise ValueError(f'ppppp {self.convert_to}.')
        else:
            spectra = torch.nan_to_num(torch.tensor(file['y'].values, requires_grad=True)).to(torch.float)
            #spectra = torch.nan_to_num(torch.tensor(np.abs(file['y'].values), requires_grad=True)).to(torch.float)

        #Reshapes it as required
        spectra = spectra.reshape(1, 1, 3106)
        
        #Prevents unknown problem with NaN from before
        func_group = torch.nan_to_num(torch.tensor(file['funcGroups'].values[self.func_group_number], requires_grad=True)).to(torch.float)

        return spectra, func_group
    
    def get_nist_id(self, index):
        return self.data.iloc[index][0]

In [5]:
#definition of neural network architecture for "simple CNN"
#NOTE: This class is  overridden in the simple_CNN_transferLearning_NIST_ENFSI notebook.
#It has been retained here for compatibility with the existing code.
class CNN(torch.nn.Module):
    '''
    Functional groups recognition CNN model.
    '''
    def __init__(self) -> None:
        super(CNN, self).__init__()

        kernel_size_1 = 5
        stride_conv_1 = 1

        stride_pool_1 = 3
        filter_size_1 = 3

        num_out_3 = 20
        num_out_4 = 10

        self.conv_1 = torch.nn.Conv1d(1, 10, kernel_size = kernel_size_1, stride = stride_conv_1)

        self.pool_1 = torch.nn.MaxPool1d(filter_size_1, stride_pool_1)

        self.conv_2 = torch.nn.Conv1d(10, 10, kernel_size = kernel_size_1, stride = stride_conv_1)

        self.pool_2 = torch.nn.MaxPool1d(filter_size_1, stride_pool_1)

        self.conv_3 = torch.nn.Conv1d(10, 10, kernel_size = kernel_size_1, stride = stride_conv_1)

        self.pool_3 = torch.nn.MaxPool1d(filter_size_1, stride_pool_1)



        self.linear_1 = torch.nn.Linear(1130, num_out_3)

        self.linear_2 = torch.nn.Linear(num_out_3, num_out_4)

        self.linear_out = torch.nn.Linear(num_out_4, 1)


    def forward(self, x):
        bs, c, h, w = x.shape
        x = x.reshape(bs, c, h * w)
        out = self.conv_1(x)
        out = F.tanh(out)
        out = self.pool_1(out)

        out = self.conv_2(out)
        out = F.tanh(out)
        out = self.pool_2(out)
        
        out = self.conv_3(out)
        out = F.tanh(out)
        out = self.pool_3(out)


        out = out.reshape(x.shape[0], 1, out.shape[2] * out.shape[1])

        
        out = self.linear_1(out)
        out = F.leaky_relu(out)

        out = self.linear_2(out)
        out = F.selu(out)

        out = self.linear_out(out)
        out = F.sigmoid(out)

        return out.squeeze(1)

